# Run a query against a natural-language logic program

Loads a program from `programs/`, turns a plain-English question into goals with Claude Haiku, and proves them with `NLEngine`, which asks Jev whenever a goal and a clause are worded differently.

Needs `TYPESAFE_API_KEY` and `CLAUDE_API_KEY` in `.env` at the project root, and the proof widget built with `pnpm build:widget`.

In [1]:
import { load } from "jsr:@std/dotenv@0.225.5";
import Anthropic from "@anthropic-ai/sdk";
import { TypeSafeClient } from "@typesafe-ai/sdk";
import { NLEngine, type ExecutionStep } from "../src/shared/engine/NLEngine.ts";
import { showLiteral } from "../src/shared/engine/program.ts";
import { createAIUnifier } from "../src/server/aiUnifier.ts";
import { readProgram } from "../src/server/programs.ts";
import { claudeCompleter, createTranslator, readQuery } from "../src/server/questionTranslator.ts";
import { ProofWidget } from "./proofWidget.ts";

await load({ envPath: "../.env", export: true });
for (const name of ["TYPESAFE_API_KEY", "CLAUDE_API_KEY"]) {
  if (!Deno.env.get(name)) throw new Error(`${name} isn't set in .env`);
}

const jev = new TypeSafeClient({ apiKey: Deno.env.get("TYPESAFE_API_KEY") });
const unify = createAIUnifier((request) => jev.systemOne(request));
const translate = createTranslator(claudeCompleter(new Anthropic({ apiKey: Deno.env.get("CLAUDE_API_KEY") })));

## Program and question

Leave `QUESTION` empty to ask the program's own `# Try:` question. A question written with variables, like `X is a grandfather of Y?`, skips translation.

In [2]:
const PROGRAM = "family";
const QUESTION = "";

const file = await readProgram("../programs", PROGRAM);
if (!file) throw new Error(`There's no programs/${PROGRAM}.nl`);
const question = QUESTION || file.query;

console.log(file.program);
console.log(`\nQuestion: ${question}`);

# The Simpson family, with facts phrased different ways
Orville is the father of Abe.
Abe is Homer's father.
Homer is the father of Bart.
Lisa's dad is Homer.
Homer is Maggie's father.
If X is the father of Y then X is a parent of Y.
X is a grandfather of Y if X is the father of Z and Z is a parent of Y.
# Try: Who are all the grandfathers?
# Try: Who is a father but not a grandfather?

Question: Who are all the grandfathers?


## Goals

In [3]:
const query = await readQuery(file.program, question, translate);

console.log(`${query.kind} question`);
for (const goals of query.alternatives) console.log("  " + goals.map(showLiteral).join(" and "));

wh question
  X is a grandfather of Y


## Proof search

Each line is one attempt to unify a goal with a clause, indented by the depth of the proof. Set `SHOW_FAILURES` to see the attempts Jev turned down too, with its confidence in each.

The widget above the log shows the proof pane from the app, a step at a time: page through it with its buttons, or the arrow keys once you've clicked it. It follows the newest step while the search runs. It's an [anywidget](https://anywidget.dev), which works in VS Code as is; JupyterLab needs `pip install anywidget` in its environment.

In [4]:
const SHOW_FAILURES = false;

const describe = (step: ExecutionStep): string | null => {
  if (step.type === "cutoff") return `… ${step.goal}: ${step.reason}`;
  if (step.type === "negation") return `${step.holds ? "✓" : "✗"} not ${step.goal}`;
  const u = step.unification;
  if (!u.unified && (!SHOW_FAILURES || u.method === "wording")) return null;
  const how = u.method === "jev" ? `jev ${u.confidence.toFixed(2)}${u.cached ? " cached" : ""}` : "wording";
  const bindings = Object.entries(u.bindings).map(([v, value]) => `${v}=${value}`).join(", ");
  return `${u.unified ? "✓" : "✗"} ${step.goal}  ~  ${step.clause.head}  [${how}]${bindings ? "  " + bindings : ""}`;
};

// A success step carries the frame it leads to, so the goal it proved sits one level up
const depth = (step: ExecutionStep) => (step.type === "success" ? step.frame.depth - 1 : step.frame.depth);

const engine = new NLEngine(file.program, unify);
const proof = await ProofWidget.create(file.program, question, query);
await Deno.jupyter.display(proof);

const started = performance.now();
const result = await engine
  .run(query, {
    onStep: (step) => {
      proof.step(step, engine.stats);
    },
  })
  .catch((error) => {
    proof.end("error", { stats: engine.stats, truncated: engine.truncated, error });
    throw error;
  });
proof.end("done", result);
const elapsedMs = Math.round(performance.now() - started);